# Clasificación Escalamiento: 8 y 16 Qubits

Este notebook está configurado para ajustar automáticamente la resolución de la imagen (vía PCA) al número de qubits seleccionados.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# === CONFIGURACIÓN ===
n_qubits = 16  # <--- CAMBIA AQUÍ A 8 O 16
# =====================

# 1. Carga de datos (Dígitos 0 y 1)
digits = datasets.load_digits()
mask = np.isin(digits.target, [0, 1])
X_raw = digits.data[mask]
y_raw = digits.target[mask]

# 2. PCA dinámico según el número de qubits
pca = PCA(n_components=n_qubits)
X_pca = pca.fit_transform(X_raw)

# Escalado
X_scaled = MinMaxScaler(feature_range=(0, 1)).fit_transform(X_pca)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=42)

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

print(f"Configurado para {n_qubits} qubits.")
print(f"Varianza explicada total: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

Configurado para 16 qubits.
Varianza explicada total: 93.49%


### Construcción del Circuito para {n_qubits} Qubits

In [2]:
from qiskit.circuit.library import zz_feature_map, real_amplitudes

# Generar circuitos dinámicamente
feature_map = zz_feature_map(n_qubits, reps=2)
ansatz = real_amplitudes(n_qubits, reps=1)

print(f"Feature Map y Ansatz creados para {n_qubits} qubits.")

Feature Map y Ansatz creados para 16 qubits.


### Entrenamiento con VQC

In [4]:
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=100),
    sampler=StatevectorSampler()
)

print("Entrenando...")
vqc.fit(X_train, y_train_oh)
print("¡Listo!")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Entrenando...


KeyboardInterrupt: 

### Evaluación Final

In [ ]:
accuracy = vqc.score(X_test, y_test_oh)
print(f"Precisión final ({n_qubits} qubits): {accuracy * 100:.2f}%")